In [11]:
import pandas as pd
import gc
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

# Optional: suppress LightGBM warnings
warnings.filterwarnings("ignore", category=UserWarning)

# 📂 Load and clean data
df = pd.read_csv('walmart-sales-dataset-of-45stores.csv')
df.dropna(inplace=True)
df = df[df['Weekly_Sales'] > 0]

# 🗓️ Feature engineering from date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df.drop('Date', axis=1, inplace=True)

# 🔤 Encode categorical variables
le = LabelEncoder()
df['Store'] = le.fit_transform(df['Store'])

# 🎯 Feature-target split
X = df.drop(['Weekly_Sales'], axis=1)
y = df['Weekly_Sales']

# 🔀 Train-validation-test split (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 📏 Scale features with column preservation
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_val = pd.DataFrame(scaler.transform(X_val), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# 🤖 Define classical models
models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(),
    'DecisionTree': DecisionTreeRegressor(),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10),
    'GradientBoosting': GradientBoostingRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
    'XGBoost': XGBRegressor(learning_rate=0.1, max_depth=6),
    'LightGBM': LGBMRegressor(num_leaves=31, learning_rate=0.05, force_row_wise=True)
}

# 📊 Train and evaluate models
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = r2_score(y_test, y_pred)
    gc.collect()

# 💾 Save results
pd.Series(results).to_csv('classical_model_results.csv')


[LightGBM] [Info] Total Bins 1112
[LightGBM] [Info] Number of data points in the train set: 3861, number of used features: 9
[LightGBM] [Info] Start training from score 1039862.835899
